In [3]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     cross_validate, GridSearchCV)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (classification_report, confusion_matrix,
                              f1_score, accuracy_score, ConfusionMatrixDisplay)
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
df = pd.read_csv("student_dropout_dataset.csv")

df["enroll_date"] = pd.to_datetime(df["enroll_date"])
df["days_since_enroll"] = (pd.Timestamp.now() - df["enroll_date"]).dt.days

drop_cols = ["student_id", "enroll_date", "label", "label_name", "label_multiclass"]
X = df.drop(columns=drop_cols)
y = df["label_multiclass"]

print("Features:", X.columns.tolist())
print("X shape:", X.shape)
print("y distribution:\n", y.value_counts().sort_index())

Features: ['age', 'region', 'exam_season', 'courses_enrolled', 'completed_assignments', 'completion_rate', 'login_frequency', 'last_activity_days_ago', 'forum_posts_count', 'dropout_score', 'days_since_enroll']
X shape: (5000, 11)
y distribution:
 label_multiclass
0    1704
1    1663
2    1633
Name: count, dtype: int64


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print("Train distribution:\n", y_train.value_counts().sort_index())
print("Test distribution:\n", y_test.value_counts().sort_index())

Train: (4000, 11), Test: (1000, 11)
Train distribution:
 label_multiclass
0    1363
1    1330
2    1307
Name: count, dtype: int64
Test distribution:
 label_multiclass
0    341
1    333
2    326
Name: count, dtype: int64


In [6]:
num_features = ["age", "courses_enrolled", "completed_assignments",
                "completion_rate", "login_frequency",
                "last_activity_days_ago", "forum_posts_count",
                "dropout_score", "days_since_enroll"]
cat_features = ["region"]
bin_features = ["exam_season"]

preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), num_features),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_features),
    ("bin", "passthrough", bin_features),
])

rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42, n_jobs=-1))
])


In [7]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = cross_validate(
    rf_pipeline, X_train, y_train,
    cv=cv,
    scoring=["accuracy", "f1_macro"],
    return_train_score=True
)

print("=== Random Forest Baseline (5-Fold CV) ===")
print(f"CV Accuracy : {cv_results['test_accuracy'].mean():.4f} "
      f"(+/- {cv_results['test_accuracy'].std():.4f})")
print(f"CV F1 Macro : {cv_results['test_f1_macro'].mean():.4f} "
      f"(+/- {cv_results['test_f1_macro'].std():.4f})")

=== Random Forest Baseline (5-Fold CV) ===
CV Accuracy : 0.9992 (+/- 0.0010)
CV F1 Macro : 0.9993 (+/- 0.0010)


In [8]:
param_grid = {
    "classifier__n_estimators": [100, 200, 300],
    "classifier__max_depth": [None, 10, 20],
    "classifier__min_samples_split": [2, 5],
    "classifier__max_features": ["sqrt", "log2"],
}

grid_search = GridSearchCV(
    rf_pipeline,
    param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring="f1_macro",       # <- optimize ด้วย macro F1
    n_jobs=-1,
    verbose=1,
    refit=True                # <- refit ด้วย best params บน X_train ทั้งหมด
)

grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print(f"Best CV F1 Macro: {grid_search.best_score_:.4f}")

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best Parameters: {'classifier__max_depth': None, 'classifier__max_features': 'sqrt', 'classifier__min_samples_split': 5, 'classifier__n_estimators': 100}
Best CV F1 Macro: 0.9995
